# Notebook 01: 2D 玩具扩散可视化

**目标**：在 2D 数据上完整跑通一遍 DDPM 训练 + 采样，把抽象的扩散公式落到看得见的地方。

**前置**：L01-L04 讲义。

**预计时间**：30 分钟（含训练）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. 准备 2D 数据集

用三种数据：
- Two moons（弯月）
- 同心环
- 螺旋

扩散模型在这些上面都能学到——视觉上能直接看到分布。

In [ ]:
def make_moons(n=5000):
    from sklearn.datasets import make_moons
    x, _ = make_moons(n, noise=0.05)
    return torch.tensor(x, dtype=torch.float32)

def make_rings(n=5000):
    theta = torch.rand(n) * 2 * np.pi
    r = torch.cat([0.5 * torch.ones(n//2), 1.0 * torch.ones(n - n//2)])
    r = r + 0.03 * torch.randn(n)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)

data = make_moons(5000)
plt.figure(figsize=(5,5))
plt.scatter(data[:,0], data[:,1], s=2, alpha=0.5)
plt.axis('equal'); plt.title('Two moons dataset'); plt.show()

## 2. 定义 DDPM schedule

linear β，T=1000

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alphas_cumprod = alphas.cumprod(0)
sqrt_a_cumprod = alphas_cumprod.sqrt()
sqrt_one_minus = (1 - alphas_cumprod).sqrt()

# Move to device
betas, alphas, alphas_cumprod = [t.to(device) for t in [betas, alphas, alphas_cumprod]]
sqrt_a_cumprod, sqrt_one_minus = sqrt_a_cumprod.to(device), sqrt_one_minus.to(device)

## 3. 可视化 forward process

看 2D 数据如何逐步变成高斯噪声。

In [ ]:
def q_sample(x0, t):
    noise = torch.randn_like(x0)
    return sqrt_a_cumprod[t].unsqueeze(-1) * x0 + sqrt_one_minus[t].unsqueeze(-1) * noise

ts_to_show = [0, 100, 300, 500, 700, 999]
fig, axes = plt.subplots(1, len(ts_to_show), figsize=(3*len(ts_to_show), 3))
for ax, t in zip(axes, ts_to_show):
    if t == 0:
        x = data.to(device)
    else:
        t_tensor = torch.full((data.shape[0],), t, device=device, dtype=torch.long)
        x = q_sample(data.to(device), t_tensor).cpu()
    ax.scatter(x[:,0].cpu(), x[:,1].cpu(), s=2, alpha=0.5)
    ax.set_title(f't={t}'); ax.axis('equal')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.tight_layout(); plt.show()

## 4. 简单 MLP 作为 noise predictor

2D 数据用 MLP 就够了——这是个绝佳的玩具实验环境。

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        # Sinusoidal position embedding
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None] * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)

class MLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.t_emb = TimeEmbedding(64)
        self.net = nn.Sequential(
            nn.Linear(2 + 64, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )
    def forward(self, x, t):
        t_emb = self.t_emb(t.float())
        return self.net(torch.cat([x, t_emb], dim=-1))

model = MLP().to(device)
print(f'params: {sum(p.numel() for p in model.parameters())/1e3:.1f}K')

## 5. 训练

标准 DDPM loss。2D + MLP 训练秒级完成。

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
data_gpu = data.to(device)
B = 512
losses = []
model.train()
for step in range(2000):
    idx = torch.randint(0, len(data_gpu), (B,))
    x0 = data_gpu[idx]
    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(x0)
    xt = sqrt_a_cumprod[t].unsqueeze(-1) * x0 + sqrt_one_minus[t].unsqueeze(-1) * noise
    eps_pred = model(xt, t)
    loss = F.mse_loss(eps_pred, noise)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 200 == 0:
        print(f'step {step}: loss = {loss:.4f}')

plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss'); plt.title('Training loss'); plt.show()

## 6. 反向采样

DDPM ancestral sampling，从 N(0, I) 一路采到目标分布。

In [ ]:
@torch.no_grad()
def sample(n=2000, return_traj=False):
    x = torch.randn(n, 2, device=device)
    traj = [x.clone()] if return_traj else None
    model.eval()
    for t in reversed(range(T)):
        t_tensor = torch.full((n,), t, device=device, dtype=torch.long)
        eps = model(x, t_tensor)
        a_t = alphas[t]; a_bar_t = alphas_cumprod[t]; b_t = betas[t]
        mean = (x - b_t / (1 - a_bar_t).sqrt() * eps) / a_t.sqrt()
        if t > 0:
            a_bar_prev = alphas_cumprod[t-1]
            var = b_t * (1 - a_bar_prev) / (1 - a_bar_t)
            x = mean + var.sqrt() * torch.randn_like(x)
        else:
            x = mean
        if return_traj and t % 100 == 0:
            traj.append(x.clone())
    return (x, traj) if return_traj else x

samples, traj = sample(return_traj=True)
samples = samples.cpu()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(data[:,0], data[:,1], s=2, alpha=0.3, label='data')
axes[0].set_title('Real'); axes[0].axis('equal'); axes[0].legend()
axes[1].scatter(samples[:,0], samples[:,1], s=2, alpha=0.3, c='r', label='generated')
axes[1].set_title('Generated'); axes[1].axis('equal'); axes[1].legend()
plt.show()

## 7. 可视化反向采样过程

看噪声如何一步步"凝聚"成数据。

In [ ]:
fig, axes = plt.subplots(1, len(traj), figsize=(2.5*len(traj), 2.5))
for i, x in enumerate(traj):
    x = x.cpu()
    axes[i].scatter(x[:,0], x[:,1], s=2, alpha=0.5)
    axes[i].set_title(f't={T - i*100 if i < len(traj)-1 else 0}')
    axes[i].axis('equal')
    axes[i].set_xlim(-3, 3); axes[i].set_ylim(-3, 3)
plt.tight_layout(); plt.show()

## 思考题

1. 把 `sample()` 中的 `t > 0` 改成 `t > 100`（提前停止），结果如何？为什么？
2. 训练时去掉 noise 预测 loss 的 `t` 采样，固定用 `t=500`，会怎样？
3. 把 MLP 改得更小（hidden=16），观察 mode collapse 是否出现